In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
import os 

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [4]:
vector_assistant.rag("How to run Kafka?")

'Based on the provided context, the question "How to run Kafka?" is not directly addressed in the given text. The text primarily focuses on Kestra, an orchestration tool, and its usage in the context of the LLM Zoomcamp course. It discusses how to configure API keys for Kestra, submit homework, and use Kestra for AI workflows, but it does not mention Kafka.\n\nHowever, since the question is about running Kafka, here\'s a general step-by-step guide to get you started:\n\n### Prerequisites\n1. **Java**: Kafka requires Java 8 or higher.\n2. **Kafka Download**: Download the Kafka binaries from the official Apache Kafka website.\n\n### Running Kafka\n1. **Extract Kafka**: Extract the downloaded Kafka binaries to a directory (e.g., `~/kafka`).\n2. **Start ZooKeeper**: Navigate to the Kafka directory and start ZooKeeper.\n   ```bash\n   # On Linux/Mac\n   bin/zookeeper-server-start.sh config/zookeeper.properties\n   # On Windows\n   .\\bin\\windows\\zookeeper-server-start.bat .\\config\\zooke

In [5]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'According to the provided context, the answer is: **Yes, you can still sign up.** \n\nAlthough the program has already begun, it is possible to join and start learning. As stated in the General Course-Related Questions, "You can also just start learning and submitting homework (while the form is open) without registering." \n\nHowever, to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [6]:
vs_index.close()